In [2]:
import os
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, LeakyReLU, Dropout, MaxPool2D, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2


In [3]:
df = pd.read_csv('train.csv')
df.head()

,ID,Class
0,377.jpg,MIDDLE
1,17814.jpg,YOUNG
2,21283.jpg,MIDDLE
3,16496.jpg,YOUNG
4,4487.jpg,MIDDLE


In [4]:
df.isnull().sum()

ID       0
Class    0
dtype: int64

In [5]:
df.describe()

,ID,Class
count,19906,19906
unique,19906,3
top,377.jpg,MIDDLE
freq,1,10804


In [6]:
import zipfile

In [7]:
"""
zip_ref = zipfile.ZipFile('./faces.zip', 'r')
zip_ref.extractall('./')
zip_ref.close()
"""

"\nzip_ref = zipfile.ZipFile('./faces.zip', 'r')\nzip_ref.extractall('./')\nzip_ref.close()\n"

In [8]:
a = os.listdir('./faces/train')
a[2]

'100.jpg'

In [9]:
from PIL import Image
import matplotlib.pyplot as plt

In [10]:
data = []
labels = []
for i in range(len(a)):
    img = Image.open('./faces/Train/' + a[i])
    img = img.resize((80,80))
    img = np.array(img)
    data.append(img)
    class_value = df[df['ID'] == a[i]]['Class'].values[0]
    if class_value == 'YOUNG':
        labels.append(0)
    elif class_value == "MIDDLE":
        labels.append(1)
    else:
        labels.append(2)
print('Done')
    

Done


In [11]:
temp = df[df['ID'] == '377.jpg']['Class'].values[0]
temp

'MIDDLE'

In [12]:
data_array = np.asarray(data)
data_array = data_array / 255.0
labels_array = np.asarray(labels)
labels_hot = to_categorical(labels_array, num_classes=3)
print(f"data: {data_array.shape}, \nhot: {labels_hot.shape}")

data: (19906, 80, 80, 3), 
hot: (19906, 3)


In [13]:
from sklearn.model_selection import train_test_split

In [14]:
xtrain, xtest, ytrain, ytest = train_test_split(data_array, labels_hot, test_size=0.1, random_state=42)

In [15]:
print(f'Train:\n{xtrain.shape}\n{ytrain.shape}\nTest:\n{xtest.shape}\n{ytest.shape}')

Train:
(17915, 80, 80, 3)
(17915, 3)
Test:
(1991, 80, 80, 3)
(1991, 3)


In [16]:
model = Sequential()

model.add(Conv2D(16, (3,3), input_shape=(80,80,3)))
#model.add(LeakyReLU(alpha=0.1))
model.add(MaxPooling2D(2,2))
model.add(Conv2D(16, (3,3)))
#model.add(LeakyReLU(alpha=0.1))
model.add(MaxPooling2D(2,2))
#model.add(Conv2D(128, (3,3)))
#model.add(Conv2D(128, (3,3)))
#model.add(Conv2D(16, (3,3), activation='relu'))
#model.add(MaxPooling2D(2,2))


model.add(Flatten())
model.add(Dense(256, activation = 'relu', kernel_regularizer=l2(0.01)))
model.add(Dense(256, activation = 'relu', kernel_regularizer=l2(0.01)))
model.add(Dense(128, activation = 'relu', kernel_regularizer=l2(0.01)))
model.add(Dense(3, activation='softmax'))


In [17]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 78, 78, 16)        448       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 39, 39, 16)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 37, 37, 16)        2320      
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 18, 18, 16)       0         
 2D)                                                             
                                                                 
 flatten (Flatten)           (None, 5184)              0         
                                                                 
 dense (Dense)               (None, 256)               1

In [18]:
model.compile(optimizer = Adam() , loss = 'categorical_crossentropy', metrics = ['acc'])

In [21]:
class MyCallback(tf.keras.callbacks.Callback):
    def epoch_on_end(self, epochs, logs={}):
        if logs.get('accuracy') > 0.9:
            print("condition satisfied")
            self.model.stop_training = True

callbacks = MyCallback()

In [22]:
model.fit(xtrain, ytrain, batch_size=10, epochs=50, validation_split=0.1, callbacks=[callbacks])

Epoch 1/50
1613/1613 [==============================] - 32s 20ms/step - loss: 0.8343 - acc: 0.6555 - val_loss: 0.8241 - val_acc: 0.6602
Epoch 2/50
1613/1613 [==============================] - 32s 20ms/step - loss: 0.8216 - acc: 0.6579 - val_loss: 0.8350 - val_acc: 0.6490
Epoch 3/50
1613/1613 [==============================] - 32s 20ms/step - loss: 0.8155 - acc: 0.6614 - val_loss: 0.8234 - val_acc: 0.6496
Epoch 4/50
1613/1613 [==============================] - 32s 20ms/step - loss: 0.8105 - acc: 0.6623 - val_loss: 0.8384 - val_acc: 0.6412
Epoch 5/50
1613/1613 [==============================] - 32s 20ms/step - loss: 0.8110 - acc: 0.6641 - val_loss: 0.8134 - val_acc: 0.6618
Epoch 6/50
1613/1613 [==============================] - 31s 19ms/step - loss: 0.8046 - acc: 0.6682 - val_loss: 0.8091 - val_acc: 0.6669
Epoch 7/50
1613/1613 [==============================] - 31s 19ms/step - loss: 0.8026 - acc: 0.6697 - val_loss: 0.8164 - val_acc: 0.6624
Epoch 8/50
1613/1613 [==========================

In [23]:
model.evaluate(xtest, ytest)

63/63 [==============================] - 1s 10ms/step - loss: 0.7795 - acc: 0.6876


[0.7794857621192932, 0.6875941753387451]